In [ ]:
!pip install ipywidgets sentence-transformers --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

%matplotlib inline

In [ ]:
import os
print("Current directory:", os.getcwd())

In [ ]:
# Load dataset — path works whether notebook is run from repo root
import pathlib
_here = pathlib.Path(os.getcwd())
_csv  = _here / "data" / "cities.csv"
if not _csv.exists():
    _csv = _here / "cities.csv"   # fallback for local runs without data/ folder
df = pd.read_csv(_csv)

In [ ]:
df["population"] = pd.to_numeric(df["population"], errors="coerce")
df["lat"]        = pd.to_numeric(df["lat"],        errors="coerce")

df = df.dropna(subset=["population", "lat"])
df = df[df["population"] > 0].copy()

df["log_pop"] = np.log10(df["population"])

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def make_axis(pos_words, neg_words):
    """Return a unit-length semantic axis from two word sets."""
    pos = model.encode(pos_words, normalize_embeddings=True).mean(axis=0)
    neg = model.encode(neg_words, normalize_embeddings=True).mean(axis=0)
    axis = pos - neg
    return axis / (np.linalg.norm(axis) + 1e-10)


def score_words(words, axis):
    """Project each word onto the axis. Returns one score per word."""
    emb = model.encode(words, normalize_embeddings=True)
    return emb @ axis

In [ ]:
# Axis 1: Global vs Local
axis1_pos = [
    "global financial hub",
    "international business center",
    "major economic capital",
    "cosmopolitan metropolis",
    "world city"
]
axis1_neg = [
    "small rural town",
    "local community center",
    "provincial town",
    "remote settlement",
    "low population town"
]

# Axis 2: Modern vs Historic
axis2_pos = [
    "modern city",
    "innovation hub",
    "high-tech metropolis",
    "smart city",
    "contemporary urban center"
]
axis2_neg = [
    "historic city",
    "ancient town",
    "medieval settlement",
    "old city center",
    "classical heritage city"
]

In [ ]:
axis1 = make_axis(axis1_pos, axis1_neg)
axis2 = make_axis(axis2_pos, axis2_neg)

In [ ]:
def centroid(words):
    emb = model.encode(words, normalize_embeddings=True)
    return emb.mean(axis=0)

def cosine_distance(a, b):
    return 1 - np.dot(a, b)

d1 = cosine_distance(centroid(axis1_pos), centroid(axis1_neg))
d2 = cosine_distance(centroid(axis2_pos), centroid(axis2_neg))

print("Axis 1 separation:", round(d1, 3), "  [need >= 0.30]") 
print("Axis 2 separation:", round(d2, 3), "  [need >= 0.30]")
assert d1 >= 0.30, "Axis 1 poles too close"
assert d2 >= 0.30, "Axis 2 poles too close"

In [ ]:
df["sem_axis1"] = score_words(df["city"].tolist(), axis1)
df["sem_axis2"] = score_words(df["city"].tolist(), axis2)

## Plot 1 — Semantic Map (Global–Local × Modern–Historic)

In [ ]:
REGION_COLORS = {
    "Europe":   "#2196F3",
    "Americas": "#FF9800",
    "Asia":     "#9C27B0",
    "Africa":   "#4CAF50",
    "Oceania":  "#F44336",
}

plt.figure(figsize=(14, 10))

for region, color in REGION_COLORS.items():
    sub = df[df["region"] == region]
    plt.scatter(sub["sem_axis1"], sub["sem_axis2"],
                label=region, color=color, alpha=0.6, s=40)

# Labels for major cities
NOTABLE = ["New York", "London", "Tokyo", "Paris", "Dubai"]
for _, row in df[df["city"].isin(NOTABLE)].iterrows():
    plt.annotate(row["city"],
                 (row["sem_axis1"], row["sem_axis2"]),
                 fontsize=8)

plt.xlabel("Local \u2190 \u2192 Global")
plt.ylabel("Historic \u2190 \u2192 Modern")
plt.title("Semantic Map of World Cities (Global\u2013Local vs Modern\u2013Historic)")
plt.grid(True)
plt.legend()

os.makedirs("figs", exist_ok=True)
plt.savefig("figs/semantic_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figs/semantic_map.png")

## Plot 2 — Geographic & Demographic Map (Latitude × Population)

In [ ]:
import matplotlib.patches as mpatches

# Region colors
REGION_COLORS_GEO = {
    "Europe":   "#1f77b4",
    "Americas": "#ff7f0e",
    "Asia":     "#9467bd",
    "Africa":   "#2ca02c",
    "Oceania":  "#d62728",
}

# Shape mapping (GaWC tiers)
def tier_group(ba):
    if ba in ("Alpha++", "Alpha+", "Alpha"):
        return "alpha_high"
    elif ba in ("Alpha-", "Beta+", "Beta"):
        return "alpha_low_beta"
    elif ba in ("Beta-", "Gamma+", "Gamma"):
        return "gamma_high"
    else:
        return "gamma_low"

SHAPE_MAP = {
    "alpha_high":     ("*", 120, "Alpha / Alpha+ / Alpha++"),
    "alpha_low_beta": ("^", 70,  "Alpha\u2212 / Beta+ / Beta"),
    "gamma_high":     ("o", 50,  "Beta\u2212 / Gamma+ / Gamma"),
    "gamma_low":      ("s", 40,  "Gamma\u2212 / Sufficiency"),
}

df["tier_key"] = df["business_activity"].apply(tier_group)

fig, ax = plt.subplots(figsize=(14, 10))

ax.grid(True, linestyle="--", alpha=0.3)
ax.axvline(0, linestyle="--", color="gray", alpha=0.6)

for tkey, (marker, size, _) in SHAPE_MAP.items():
    for region, color in REGION_COLORS_GEO.items():
        sub = df[(df["tier_key"] == tkey) & (df["region"] == region)]
        if sub.empty:
            continue
        ax.scatter(
            sub["lat"], sub["log_pop"],
            c=color, marker=marker, s=size,
            alpha=0.8, edgecolors="white", linewidths=0.5
        )

# Labels
NOTABLE_GEO = [
    "New York City", "London", "Tokyo", "Paris", "Dubai",
    "Shanghai", "Beijing", "Delhi", "Mexico City",
    "Sydney", "Buenos Aires", "Johannesburg", "Cairo",
    "Moscow", "Amsterdam", "Reykjavik"
]
for _, row in df[df["city"].isin(NOTABLE_GEO)].iterrows():
    ax.annotate(row["city"],
                (row["lat"], row["log_pop"]),
                xytext=(3, 3), textcoords="offset points", fontsize=8)

ax.set_xlabel("Latitude  \u2190  Southern Hemisphere  |  Northern Hemisphere  \u2192", fontsize=11)
ax.set_ylabel("Population  \u2190  Small City  |  Megacity  \u2192", fontsize=11)
ax.set_title(
    "World Cities \u2014 Geographic & Demographic Map\n"
    "X: Latitude   \u00b7   Y: Population (log scale)   \u00b7   Color: Region   \u00b7   Shape: GaWC Tier",
    fontsize=14, fontweight="bold"
)

# Y-axis log ticks (human-readable)
ax.set_yticks([2, 3, 4, 5, 6, 7])
ax.set_yticklabels(["100", "1K", "10K", "100K", "1M", "10M"])

# Region legend — top-right OUTSIDE plot area
region_handles = [mpatches.Patch(color=c, label=r)
                  for r, c in REGION_COLORS_GEO.items()]
leg1 = ax.legend(handles=region_handles, title="Region",
                 bbox_to_anchor=(1.01, 1.0), loc="upper left",
                 fontsize=9, title_fontsize=10, framealpha=0.95)
ax.add_artist(leg1)

# Tier legend — bottom-right OUTSIDE plot area
shape_handles = [
    plt.scatter([], [], marker=v[0], c="gray", s=v[1], label=v[2])
    for v in SHAPE_MAP.values()
]
ax.legend(handles=shape_handles, title="GaWC Business Tier",
          bbox_to_anchor=(1.01, 0.0), loc="lower left",
          fontsize=9, title_fontsize=10, framealpha=0.95)

# Quadrant labels
ax.text(-45, 7.35, "Southern & Large",   fontsize=9, alpha=0.6)
ax.text( 45, 7.35, "Northern & Large",   fontsize=9, alpha=0.6)
ax.text(-45, 1.85, "Southern & Small",   fontsize=9, alpha=0.6)
ax.text( 45, 1.85, "Northern & Small",   fontsize=9, alpha=0.6)

plt.tight_layout(rect=[0, 0, 0.83, 1])
plt.savefig("figs/geographic_map.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: figs/geographic_map.png")

## Observations

The semantic map reveals a clear separation between globally influential cities and more locally oriented ones along the horizontal axis. While the axes separate cities along conceptual dimensions, there is still noticeable overlap between regions, suggesting that geographic location alone does not fully determine a city's global or modern characteristics. Cities such as New York, London, and Tokyo appear strongly on the global side, reflecting their role in international finance and trade.

The vertical axis captures the distinction between modern and historically oriented cities. Cities associated with technological innovation and economic development tend to appear higher, while historically rich cities appear lower.

A surprising observation is that cities like Paris and Rome appear closer to the modern side despite their historical identities. This suggests that the embedding captures how cities are described in modern contexts rather than purely their historical background.

One limitation of the current axes is that they do not capture population size or geographic location. A third axis could represent economic inequality, population density, or climate, which may provide additional insights into the structure of global cities.

In [ ]:
print("\nMost Global Cities:")
display(df.nlargest(5, "sem_axis1")[["city", "region"]])

print("\nMost Local Cities:")
display(df.nsmallest(5, "sem_axis1")[["city", "region"]])